In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv1D, MaxPooling1D,
    Dense, Flatten
)
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split


In [ ]:
LABEL_MAP = {
    "_corr": 0,   # Correct
    "_id": 1,     # Insufficient depth
    "_rb": 2,     # Rounded back
    "_toe": 3     # Weight on toes
}

NUM_CLASSES = 4




In [ ]:
DATA_ROOT = "..."

X, y = [], []

for root, _, files in os.walk(DATA_ROOT):
    for file in files:
        if not file.endswith(".npy"):
            continue

        file_path = os.path.join(root, file)

        # infer label from folder name
        label_found = False
        for suffix, label_id in LABEL_MAP.items():
            if suffix in root:
                data = np.load(file_path)      # (50,17,2)
                data = data.reshape(50, 34)    # (50,34)
                X.append(data)
                y.append(label_id)
                label_found = True
                break

        if not label_found:
            print("Label not found for:", file_path)

X = np.array(X)
y = to_categorical(y, NUM_CLASSES)

print("Loaded samples:", len(X))
print("X shape:", X.shape)
print("y shape:", y.shape)



In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y.argmax(axis=1)
)


In [ ]:
input_layer = Input(shape=(50, 34))

x = Conv1D(32, kernel_size=3, activation="relu")(input_layer)  # (48, 32)
x = MaxPooling1D(pool_size=2)(x)                               # (24, 32)
x = Flatten()(x)
x = Dense(32, activation="relu")(x)

output = Dense(4, activation="softmax")(x)

model_cnn = Model(input_layer, output)


In [ ]:
model_cnn.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model_cnn.summary()

In [ ]:
history_cnn = model_cnn.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=30,
    batch_size=16,
    verbose=1
)


In [ ]:
model_cnn.save(
    "1D_CNN_Posture.keras"
)



In [ ]:

)
